# Red Team Eval — VS Code + Colab (GPU)

Runs the evaluation pipeline on Colab's GPU using Ollama. Edited in VS Code, executed on the Colab cloud runtime.

**How this differs from `colab_eval.ipynb`:**
`colab_eval.ipynb` stores the project on Google Drive and mounts it at runtime.
This notebook skips Drive entirely: you upload files directly to the Colab session via the VS Code Colab extension.
Logs are saved to `/content/logs/` inside the session and must be downloaded manually when the run finishes.

**Before running:**
1. Set runtime to GPU: *Runtime → Change runtime type → T4 GPU*
2. In the VSCode Explorer, right-click `eval.py` → *Upload to Colab* and right-click the `data/` folder → *Upload to Colab*
3. Run the cells top to bottom. Edit the **Eval configuration** cell as needed.

In [15]:
# ── Verify project files ───────────────────────────────────────────────────────
# Right-click eval.py and data/ in the VSCode Explorer and "Upload to Colab" first.
import pathlib

PROJECT_PATH = pathlib.Path('/content')

missing = [f for f in ['eval.py', 'data'] if not (PROJECT_PATH / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing in /content: {missing}\n"
        "Right-click them in the VSCode Explorer and choose 'Upload to Colab'."
    )

print('Project path:', PROJECT_PATH)
print('Files ready:', sorted(p.name for p in PROJECT_PATH.iterdir() if p.name not in ('sample_data',)))

Project path: /content
Files ready: ['.config', 'config.py', 'data', 'eval.py']


In [16]:
# ── GPU check ──────────────────────────────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null \
    && echo 'GPU OK' \
    || echo 'WARNING: No GPU detected — change runtime type before continuing'

Tesla T4, 15360 MiB
GPU OK


In [24]:
# ── Eval configuration ─────────────────────────────────────────────────────────
# Edit these values. This cell writes config.py to /content/ so eval.py picks them up.

DATASET = "10_sample"       # "all" | "1_sample" | "10_sample" | "100_sample" | "300_sample"

MODELS = [
    "openai/gemma3:270m",
    # "openai/deepseek-r1:1.5b",
    # "openai/mistral:7b",
    # "openai/MichelRosselli/apertus:8b-instruct-2509-q4_k_m",
]

GRADER_MODEL = [
    # "openai/gemma3:270m",
    # "openai/deepseek-r1:1.5b",
    "openai/mistral:7b",
    # "openai/MichelRosselli/apertus:8b-instruct-2509-q4_k_m",
]

LANGUAGES = [
    "en",
    # "de",
    # "ar",
    # "ch",
    # "ta",
]

N_RUNS = 1
CONFIG        = {"seed": 42, "temperature": 0.0, "top_p": 1.0, "max_tokens": 500}
GRADER_CONFIG = {"seed": 42, "temperature": 0.0, "top_p": 1.0, "max_tokens": 300}

# ── Write config.py ────────────────────────────────────────────────────────────
import json, textwrap, pathlib

_cfg = textwrap.dedent(f"""\
from dotenv import load_dotenv
load_dotenv()

DATASET       = {json.dumps(DATASET)}
MODELS        = {json.dumps(MODELS, indent=4)}
GRADER_MODEL  = {json.dumps(GRADER_MODEL, indent=4)}
LANGUAGES     = {json.dumps(LANGUAGES, indent=4)}
N_RUNS        = {json.dumps(N_RUNS, indent=4)}
CONFIG        = {json.dumps(CONFIG, indent=4)}
GRADER_CONFIG = {json.dumps(GRADER_CONFIG, indent=4)}
""")

config_path = pathlib.Path('/content/config.py')
config_path.write_text(_cfg)
print('config.py written to', config_path)

config.py written to /content/config.py


In [18]:
# ── Install Python dependencies ────────────────────────────────────────────────
!pip install -q inspect-ai openai deep-translator python-dotenv

In [19]:
# ── Install Ollama ─────────────────────────────────────────────────────────────
!sudo apt-get update -qq && sudo apt-get install -y -q zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 98 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [20]:
# ── Start Ollama service ───────────────────────────────────────────────────────
import subprocess, time, os

env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'

ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env,
)
time.sleep(5)
print('Ollama running')

Ollama running


In [25]:
# ── Pull required models ───────────────────────────────────────────────────────
import subprocess

existing_raw = subprocess.check_output(['ollama', 'list'], text=True)
existing = [line.split()[0] for line in existing_raw.strip().split('\n')[1:] if line.strip()]

for model in set(MODELS + GRADER_MODEL):
    base = model.removeprefix('openai/')
    if base in existing:
        print(f'  {base} already present, skipping')
    else:
        print(f'  Pulling {base}...')
        subprocess.run(['ollama', 'pull', base], check=True)

subprocess.run(['ollama', 'ps'])

  mistral:7b already present, skipping
  gemma3:270m already present, skipping


CompletedProcess(args=['ollama', 'ps'], returncode=0)

In [22]:
# ── Point to local Ollama ──────────────────────────────────────────────────────
import os
os.environ['OPENAI_API_KEY']  = 'ollama'
os.environ['OPENAI_BASE_URL'] = 'http://localhost:11434/v1'

In [26]:
# ── Run evaluation ─────────────────────────────────────────────────────────────
import subprocess, os, sys

result = subprocess.run(
    [sys.executable, 'eval.py'],
    cwd='/content',
    env=os.environ.copy(),
    capture_output=True,
    text=True,
)

print('Exit code:', result.returncode)
if result.stdout:
    print('\n--- Standard Output ---')
    print(result.stdout)
if result.stderr:
    print('\n--- Standard Error ---')
    print(result.stderr)

Exit code: 0

--- Standard Output ---
╭──────────────────────────────────────────────────────────────────────────────╮
│redteam_eval (10 samples): openai/gemma3:270m                                 │
╰──────────────────────────────────────────────────────────────────────────────╯
language: en, grader: openai/mistral:7b, dataset: en                            
total time:                  0:01:11                                            
openai/gemma3:270m           4,612 tokens [I: 236, O: 4,376]                    
openai/mistral:7b            11,088 tokens [I: 10,059, O: 1,029]                
                                                                                
yes_no_grader                                                                   
accuracy       0.700                                                            
mean           0.700                                                            
                                                                       